# UAV Route Visualization

This notebook contains the code to load UAV route data from a CSV file, parse the coordinate information, and visualize the flight paths in both 3D and 2D.

In [1]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.colors import qualitative
from mpl_toolkits.mplot3d import Axes3D

# Load the data
file_path = './logs/python_routes_received.csv'
df = pd.read_csv(file_path)

print(f"Loaded {len(df)} waypoints for {df['plan_id'].nunique()} flight plans.")
df.head()

Loaded 5492 waypoints for 181 flight plans.


,plan_id,start_node_id,end_node_id,waypoint_index,waypoint_node_id,overfly_time_sim_s,segment_duration_s,cumulative_duration_s,planned_total_duration_s,start_time_sim_s,planned_completion_time_sim_s,waypoint_speed_mps,pathfinding_duration_s,total_processing_time_s,logged_at_unix_s
0,FP000002,L0_X0_Y2,L0_X6_Y4,0,L0_X0_Y2,47.000000,0.000000,0.000000,336.774305,47.0,383.774305,25,0.053965,0.14169,1.772647e+09
1,FP000002,L0_X0_Y2,L0_X6_Y4,1,L1_X0_Y2,48.219200,1.219200,1.219200,336.774305,47.0,383.774305,25,0.053965,0.14169,1.772647e+09
2,FP000002,L0_X0_Y2,L0_X6_Y4,2,L2_X0_Y2,49.438400,1.219200,2.438400,336.774305,47.0,383.774305,25,0.053965,0.14169,1.772647e+09
3,FP000002,L0_X0_Y2,L0_X6_Y4,3,L2_X1_Y2,65.433240,15.994840,18.433240,336.774305,47.0,383.774305,25,0.053965,0.14169,1.772647e+09
4,FP000002,L0_X0_Y2,L0_X6_Y4,4,L2_X2_Y2,81.428089,15.994849,34.428089,336.774305,47.0,383.774305,25,0.053965,0.14169,1.772647e+09


## 1. Parse Coordinates

The `waypoint_node_id` column contains the level ($L$), $X$, and $Y$ coordinates in the string format `L#_X#_Y#`. We will extract these into separate numeric columns.

In [2]:
def parse_node_id(node_id):
    # Format: L{level}_X{x}_Y{y}
    match = re.search(r'L(\d+)_X(\d+)_Y(\d+)', node_id)
    if match:
        return int(match.group(1)), int(match.group(2)), int(match.group(3))
    return None, None, None

# Apply parsing logic
df[['L', 'X', 'Y']] = df['waypoint_node_id'].apply(lambda x: pd.Series(parse_node_id(x)))
df[['waypoint_node_id', 'L', 'X', 'Y']].head()

,waypoint_node_id,L,X,Y
0,L0_X0_Y2,0,0,2
1,L1_X0_Y2,1,0,2
2,L2_X0_Y2,2,0,2
3,L2_X1_Y2,2,1,2
4,L2_X2_Y2,2,2,2


## 2. 3D Route Visualization

This plot shows the routes in 3D space, incorporating flight levels as the Z-axis.

In [3]:
# Select unique plans to display (e.g., first 10)
# plans_to_plot = df['plan_id'].unique()[:10]
plans_to_plot = ['FP000174', 'FP000187']

rng_3d = np.random.default_rng(42)
label_jitter_xy = 0.15
label_jitter_z = 0.06
color_cycle = qualitative.Plotly

fig3d = go.Figure()

for i, plan_id in enumerate(plans_to_plot):
    plan_data = df[df['plan_id'] == plan_id].sort_values(by='waypoint_index')
    route_color = color_cycle[i % len(color_cycle)]

    fig3d.add_trace(
        go.Scatter3d(
            x=plan_data['X'],
            y=plan_data['Y'],
            z=plan_data['L'],
            mode='lines+markers',
            name=plan_id,
            legendgroup=plan_id,
            marker=dict(size=4, color=route_color),
            line=dict(width=4, color=route_color),
            opacity=0.85
        )
    )

    label_x = []
    label_y = []
    label_z = []
    label_text = []
    for _, row in plan_data.iterrows():
        label_x.append(row['X'] + rng_3d.uniform(-label_jitter_xy, label_jitter_xy))
        label_y.append(row['Y'] + rng_3d.uniform(-label_jitter_xy, label_jitter_xy))
        label_z.append(row['L'] + rng_3d.uniform(-label_jitter_z, label_jitter_z))
        label_text.append(f"{row['overfly_time_sim_s']:.1f}s")

    fig3d.add_trace(
        go.Scatter3d(
            x=label_x,
            y=label_y,
            z=label_z,
            mode='text',
            text=label_text,
            textfont=dict(size=10, color=route_color),
            showlegend=False,
            legendgroup=plan_id,
            hoverinfo='skip'
        )
    )

labels_on_visibility = []
labels_off_visibility = []
for _ in plans_to_plot:
    labels_on_visibility.extend([True, True])
    labels_off_visibility.extend([True, False])

fig3d.update_layout(
    title='UAV Routes (3D View) - Interactive',
    scene=dict(
        xaxis_title='X Coordinate',
        yaxis_title='Y Coordinate',
        zaxis_title='Flight Level (L)'
    ),
    legend=dict(title='Routes (click to toggle)', groupclick='togglegroup'),
    margin=dict(l=0, r=0, t=50, b=0),
    updatemenus=[
        dict(
            type='buttons',
            direction='left',
            x=0.0,
            y=1.12,
            buttons=[
                dict(label='Labels ON', method='update', args=[{'visible': labels_on_visibility}]),
                dict(label='Labels OFF', method='update', args=[{'visible': labels_off_visibility}])
            ]
        )
    ]
)

fig3d.show()

## 3. Top-down 2D Visualization

A standard 2D view focusing on the $X$ and $Y$ layout.

In [4]:
rng_2d = np.random.default_rng(43)
label_jitter_2d = 0.15
color_cycle = qualitative.Plotly

fig2d = go.Figure()

for i, plan_id in enumerate(plans_to_plot):
    plan_data = df[df['plan_id'] == plan_id].sort_values(by='waypoint_index')
    route_color = color_cycle[i % len(color_cycle)]

    fig2d.add_trace(
        go.Scatter(
            x=plan_data['X'],
            y=plan_data['Y'],
            mode='lines+markers',
            name=plan_id,
            legendgroup=plan_id,
            marker=dict(size=7, color=route_color),
            line=dict(width=3, color=route_color),
            opacity=0.85
        )
    )

    label_x = []
    label_y = []
    label_text = []
    for _, row in plan_data.iterrows():
        label_x.append(row['X'] + rng_2d.uniform(-label_jitter_2d, label_jitter_2d))
        label_y.append(row['Y'] + rng_2d.uniform(-label_jitter_2d, label_jitter_2d))
        label_text.append(f"{row['overfly_time_sim_s']:.1f}s")

    fig2d.add_trace(
        go.Scatter(
            x=label_x,
            y=label_y,
            mode='text',
            text=label_text,
            textfont=dict(size=11, color=route_color),
            showlegend=False,
            legendgroup=plan_id,
            hoverinfo='skip'
        )
    )

labels_on_visibility = []
labels_off_visibility = []
for _ in plans_to_plot:
    labels_on_visibility.extend([True, True])
    labels_off_visibility.extend([True, False])

fig2d.update_layout(
    title='UAV Routes (Top-down 2D View) - Interactive',
    xaxis_title='X Coordinate',
    yaxis_title='Y Coordinate',
    legend=dict(title='Routes (click to toggle)', groupclick='togglegroup'),
    template='plotly_white',
    updatemenus=[
        dict(
            type='buttons',
            direction='left',
            x=0.0,
            y=1.12,
            buttons=[
                dict(label='Labels ON', method='update', args=[{'visible': labels_on_visibility}]),
                dict(label='Labels OFF', method='update', args=[{'visible': labels_off_visibility}])
            ]
        )
    ]
)

fig2d.show()